Rimozione del modulo numpy in favore di una versione più vecchia a causa di errori di compatibilità.\
Dopo l'installazione sarà necessario riavviare il *Runtime*

In [ ]:
!pip uninstall numpy -y
!pip3 install mxnet-mkl==1.6.0 numpy==1.23.1

Installazione del modulo *onnx*

In [ ]:
!pip install onnx

Verifica delle installazioni.\
Verificato funzionamento per:\
*mxnet-mkl==1.6.0\
numpy==1.23.1\
onnx==1.17.0*

In [ ]:
!pip freeze | grep mxnet
!pip freeze | grep numpy
!pip freeze | grep onnx

Import delle librerie necessarie

In [ ]:
import mxnet as mx
import matplotlib.pyplot as plt
import numpy as np
from collections import namedtuple
from mxnet.gluon.data.vision import transforms
from mxnet.contrib.onnx.onnx2mx.import_model import import_model
import os

Importazione delle risorse (immagini, modello, etichette)

In [ ]:
mx.test_utils.download('https://github.com/onnx/models/raw/main/validated/vision/classification/squeezenet/model/squeezenet1.1-7.onnx')
mx.test_utils.download('https://s3.amazonaws.com/model-server/inputs/kitten.jpg')
mx.test_utils.download('https://s3.amazonaws.com/onnx-model-zoo/synset.txt')


Inizializzazione etichette e modello

In [ ]:
with open('synset.txt', 'r') as f:
    labels = [l.rstrip() for l in f]

# Enter path to the ONNX model file
model_path = 'squeezenet1.1-7.onnx'
sym, arg_params, aux_params = import_model(model_path)

Dichiarazione di funzioni

In [ ]:
Batch = namedtuple('Batch', ['data'])
def get_image(path, show=False):
    img = mx.image.imread(path)
    if img is None:
        return None
    if show:
        plt.imshow(img.asnumpy())
        plt.axis('off')
    return img

In [ ]:
def preprocess(img):
    transform_fn = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    img = transform_fn(img)
    img = img.expand_dims(axis=0)
    return img

In [ ]:
def predict(path):
    img = get_image(path, show=True)
    img = preprocess(img)
    mod.forward(Batch([img]))
    # Take softmax to generate probabilities
    scores = mx.ndarray.softmax(mod.get_outputs()[0]).asnumpy()
    # print the top-5 inferences class
    scores = np.squeeze(scores)
    a = np.argsort(scores)[::-1]
    for i in a[0:5]:
        print('%f class=%s ; probability=%f' %(i,labels[i],scores[i]))

Funzione utilizzata per misurare il tempo di esecuzione su *n* esecuzioni

In [ ]:
def timedPredict(img):
    for i in range(1000):
      mod.forward(Batch([img]))


Determinazione e impostazione del contesto

In [ ]:
# Determine and set context
if len(mx.test_utils.list_gpus())==0:
    ctx = mx.cpu()
else:
    ctx = mx.gpu(0)
# Load module
mod = mx.mod.Module(symbol=sym, context=ctx, label_names=None)
mod.bind(for_training=False, data_shapes=[('data', (1,3,224,224))],
         label_shapes=mod._label_shapes)
mod.set_params(arg_params, aux_params, allow_missing=True, allow_extra=True)

Test di inferenza sul modello

In [ ]:
# Enter path to the inference image below
# img_path = 'dog.jpg'
img_path = 'kitten.jpg'
predict(img_path)

I due bolocchi successivi sono utilizzati per eseguire un'inferenza sul modello effettuando *n* ripetizioni per valutarne il tempo di esecuzione

In [ ]:
img_path = 'kitten.jpg'
img = get_image(img_path, show=True)
img = preprocess(img)


In [ ]:
%%timeit -r 10 -n 10
timedPredict(img)